In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import numpy as np

from src.utils import (
    get_args,
    set_seed,
    get_datesets_and_loaders,
    get_trained_VAE,
    get_trained_VAE_with_domain_classifier,
    get_trained_classifier,
    get_trained_classifier_Base,
    test_model,
    prepare_report,
    run_all_senario,
)
from src.tupl import run_tupl
from src.our_tupl import GENERATION_POLICIES, run_m1_tupl, run_all_senario_m1_tupl

/home/asad/workspace/DomainProject/changeDomain/notebooks/effective-gzsda/gzsda/src/utils.py:3: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.1)
  import scipy
/home/asad/workspace/anaconda3/envs/asad/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
STYLE_LABELS = [
    "angry", "childlike", "depressed", "neutral",
    "old", "proud", "strutting",
]
DOMAIN_SET = [f"all_but_{s}" for s in STYLE_LABELS] + STYLE_LABELS
PAIRS = [(i, i + len(STYLE_LABELS)) for i in range(len(STYLE_LABELS))]
DATA_DIR = "./data/ActionStyleDataset_v2/"
DATASET_DETAILS = {
    "prefix": "ActionStyle-",
    "suffix": "-clip.mat",
    "resnet_feature": "clip_features",
    "split_file_name": "instanceSplit_actionStyle_v2_unseen2.mat",
}
NUM_LABELS = 5

print("DOMAIN_SET", DOMAIN_SET)
print("PAIRS", [(DOMAIN_SET[s], DOMAIN_SET[t]) for s, t in PAIRS])

DOMAIN_SET ['all_but_angry', 'all_but_childlike', 'all_but_depressed', 'all_but_neutral', 'all_but_old', 'all_but_proud', 'all_but_strutting', 'angry', 'childlike', 'depressed', 'neutral', 'old', 'proud', 'strutting']
PAIRS [('all_but_angry', 'angry'), ('all_but_childlike', 'childlike'), ('all_but_depressed', 'depressed'), ('all_but_neutral', 'neutral'), ('all_but_old', 'old'), ('all_but_proud', 'proud'), ('all_but_strutting', 'strutting')]


In [4]:
import json
from pathlib import Path

RESULT_OBJ_PATH = "./result/json/actionStyle_v2.json"
RESULT_CSV_PATH = "./result/csv/actionStyle_v2.csv"
path = Path(RESULT_OBJ_PATH)

if path.exists():
    with path.open("r", encoding="utf-8") as f:
        result = json.load(f)
else:
    result = {}

result.keys()

dict_keys(['base', 'CCVAE', 'our0', 'our_GRE', 'TUPL', 'our_TUPL_real_plus_src2tgt', 'our_TUPL_real_plus_src2tgt_unseen', 'our_TUPL_interp_src2tgt'])

In [5]:
base = "base"
CCVAE = "CCVAE"
our0 = "our0"
our_GRE = "our_GRE"
tupl = "TUPL"
our_tupl = "our_TUPL"

# clear last result
# result.pop(base, None)
# result.pop(CCVAE, None)
# result.pop(our0, None)
# result.pop(our_GRE, None)
# result.pop(tupl, None)
# for k in [f"{our_tupl}_{p}" for p in GENERATION_POLICIES]:
#     result.pop(k, None)


## Base


In [6]:
def main_base(args):
    set_seed(args)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    classifier = get_trained_classifier_Base(
        data_loaders=data_loaders,
        NUM_LABELS=NUM_LABELS,
        device=device,
        input_dim=512)

    return test_model(classifier, datasets["test"], data_loaders["test"], device)


In [7]:
if base not in result:
    result[base] = run_all_senario(main_base, DOMAIN_SET, input_dim=512, num_trial=6, pairs=PAIRS)


# GZSDA


In [8]:
def main_gzsda(args):
    set_seed(args)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE(
        data_loaders=data_loaders,
        args=args,
        device=device)

    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device,
        input_dim=512)

    return test_model(classifier, datasets["test"], data_loaders["test"], device)


In [9]:
if CCVAE not in result:
    result[CCVAE] = run_all_senario(main_gzsda, DOMAIN_SET, input_dim=512, num_trial=6, pairs=PAIRS)


## m0


In [10]:
def main_m0(args):
    set_seed(args)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE(
        data_loaders=data_loaders,
        args=args,
        device=device)

    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device,
        change_policy_epoch=30,
        input_dim=512)

    return test_model(classifier, datasets["test"], data_loaders["test"], device)


In [11]:
if our0 not in result:
    result[our0] = run_all_senario(main_m0, DOMAIN_SET, input_dim=512, num_trial=6, pairs=PAIRS)


## m1: seperate after encoder


In [12]:
def main_m1(args):
    set_seed(args)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE_with_domain_classifier(
        data_loaders=data_loaders,
        args=args,
        device=device)

    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device,
        change_policy_epoch=30,
        input_dim=512)

    return test_model(classifier, datasets["test"], data_loaders["test"], device)


In [13]:
if our_GRE not in result:
    result[our_GRE] = run_all_senario(main_m1, DOMAIN_SET, input_dim=512, num_trial=6, pairs=PAIRS)


## TUPL

Leave-one-style-out: source is `all_but_X` (six styles concatenated), target is `X`.

In [14]:
def main_tupl(args):
    set_seed(args)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    acc_s, acc_u, h = run_tupl(
        data_root="./data/",
        dataset="actionstyle_v2",
        source=args.sourceDomainIndex,
        target=args.targetDomainIndex,
        trial=args.trialIndex,
        seed=args.seed,
        device=device,
        quiet=True,
        return_model=False,
    )
    print("seen acc:{:2.4f}, unseen acc:{:2.4f}, H:{:2.4f}".format(acc_s / 100, acc_u / 100, h / 100))
    return None, None, acc_s / 100.0, acc_u / 100.0

In [15]:
if tupl not in result:
    result[tupl] = run_all_senario(
        main_tupl, DOMAIN_SET, input_dim=512, num_trial=6, pairs=PAIRS
    )


## our_TUPL: m1 VAE + TUPL


In [16]:
def main_m1_tupl(args, policy="real_plus_src2tgt"):
    set_seed(args)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    acc_s, acc_u, h = run_m1_tupl(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS,
        policy=policy,
        device=device,
        quiet=True,
    )
    print("seen acc:{:2.4f}, unseen acc:{:2.4f}, H:{:2.4f}".format(acc_s / 100, acc_u / 100, h / 100))
    return None, None, acc_s / 100.0, acc_u / 100.0


In [17]:
missing = [p for p in GENERATION_POLICIES if f"{our_tupl}_{p}" not in result]
if missing:
    result.update(run_all_senario_m1_tupl(
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS,
        policies=missing,
        input_dim=512,
        num_trial=6,
        pairs=PAIRS,
    ))


## Merge results

In [18]:
with open(RESULT_OBJ_PATH, "w") as f:
    json.dump(result, f, indent=2)

In [19]:
# ignore our0
_ = result.pop(our0, None)
_ = result.pop("our_TUPL_real_plus_src2tgt", None)
_ = result.pop("our_TUPL_real_plus_src2tgt_unseen", None)
# _ = result.pop("our_TUPL_interp_src2tgt", None)

In [20]:
import pandas as pd
import re

rows = [(k, m, result[m][k]) for m in result for k in result[m]]
df = pd.DataFrame(rows, columns=["domain", "method", "values"])

def extract_metrics(text):
    matches = dict(re.findall(r"([\w-]+):\s+([\d.]+\s*±\s*[\d.]+)", text))
    return pd.Series(matches)

df[["seen", "unseen", "H-mean"]] = df["values"].apply(extract_metrics)
df = df[["domain", "method", "seen", "unseen", "H-mean"]]

df["method"] = pd.Categorical(
    df["method"],
    categories=[base, CCVAE, our0, our_GRE, tupl] + [f"{our_tupl}_{p}" for p in GENERATION_POLICIES],
    ordered=True,
)
df = df.sort_values(["domain", "method"]).reset_index(drop=True)
df


,domain,method,seen,unseen,H-mean
0,all_but_angry -> angry,base,100.00 ± 0.00,91.67 ± 2.58,95.56 ± 1.40
1,all_but_angry -> angry,CCVAE,100.00 ± 0.00,85.52 ± 4.35,91.90 ± 2.53
2,all_but_angry -> angry,our_GRE,99.07 ± 0.93,94.10 ± 1.65,96.48 ± 1.00
3,all_but_angry -> angry,TUPL,94.79 ± 4.09,45.24 ± 5.09,60.57 ± 4.98
4,all_but_angry -> angry,our_TUPL_interp_src2tgt,94.70 ± 2.75,51.70 ± 8.17,64.45 ± 7.42
5,all_but_childlike -> childlike,base,95.83 ± 2.08,91.18 ± 2.11,93.28 ± 1.12
6,all_but_childlike -> childlike,CCVAE,94.44 ± 2.73,84.33 ± 3.56,88.76 ± 2.05
7,all_but_childlike -> childlike,our_GRE,93.06 ± 3.79,95.36 ± 0.74,94.01 ± 2.16
8,all_but_childlike -> childlike,TUPL,90.05 ± 2.75,36.11 ± 4.63,50.61 ± 4.90
9,all_but_childlike -> childlike,our_TUPL_interp_src2tgt,91.32 ± 4.11,55.65 ± 8.87,65.93 ± 6.19


In [21]:
Path(RESULT_CSV_PATH).parent.mkdir(parents=True, exist_ok=True)
df.to_csv(RESULT_CSV_PATH, index=False)